# ExomSystem — Consultas SQL en Jupyter
Usamos `jupysql` para escribir SQL puro en celdas con `%%sql`.

> **Requisito:** `docker-compose up --build` corriendo antes de ejecutar.

## 0. Conexión a PostgreSQL

In [1]:
import sys
import os
from sqlalchemy import create_engine

# Asegura que Python encuentra los módulos del proyecto desde /app
sys.path.insert(0, "/app")

# Cargar extension jupysql
%load_ext sql

# Leer variables de entorno que inyecta docker-compose
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "civictech_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASSWORD", "")

# Crear engine SQLAlchemy y pasarlo directamente a jupysql
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

%sql engine

print(f"Conectado a {DB_NAME} en {DB_HOST}:{DB_PORT}")

There's a new jupysql version available (0.11.1), you're running 0.10.10. To upgrade: pip install jupysql --upgrade
Deploy Streamlit apps for free on Ploomber Cloud! Learn more: https://ploomber.io/s/signup
Conectado a civictech_db en db:5432


## 1. Verificar que PostGIS está activo

In [2]:
%%sql
SELECT PostGIS_Full_Version();

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

KeyError: 'DEFAULT'

## 2. Ver las tablas del schema

In [3]:
%%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

6 rows affected.

KeyError: 'DEFAULT'

## 3. Consultas sobre usuarios

In [ ]:
%%sql
SELECT id_usuario, nombre_completo, email, reputacion_score
FROM usuarios
ORDER BY reputacion_score DESC;

## 4. Consultas sobre reportes

In [ ]:
%%sql
-- Todos los reportes con nombre de usuario y tipo de infracción
SELECT
    r.id_reporte,
    u.nombre_completo,
    ti.descripcion         AS tipo_infraccion,
    r.patente_vehiculo,
    r.estado_resolucion,
    r.fecha_hora_servidor
FROM reportes r
JOIN usuarios          u  ON u.id_usuario = r.id_usuario
JOIN tipos_infraccion  ti ON ti.id_tipo   = r.id_tipo
ORDER BY r.fecha_hora_servidor DESC;

In [ ]:
%%sql
-- Solo los reportes pendientes
SELECT id_reporte, patente_vehiculo, fecha_hora_servidor
FROM reportes
WHERE estado_resolucion = 'Pendiente'
ORDER BY fecha_hora_servidor DESC;

## 5. Consultas espaciales PostGIS

In [ ]:
%%sql
-- Coordenadas GPS en formato legible (lon, lat)
SELECT
    id_reporte,
    patente_vehiculo,
    ST_X(coordenadas_gps) AS longitud,
    ST_Y(coordenadas_gps) AS latitud,
    estado_resolucion
FROM reportes;

In [ ]:
%%sql
-- Reportes en un radio de 500 metros desde el centro de Chilecito
-- Chilecito: lon=-66.8541, lat=-29.4967
SELECT
    id_reporte,
    patente_vehiculo,
    ST_X(coordenadas_gps)                                        AS longitud,
    ST_Y(coordenadas_gps)                                        AS latitud,
    ROUND(ST_Distance(
        coordenadas_gps::geography,
        ST_SetSRID(ST_MakePoint(-66.8541, -29.4967), 4326)::geography
    )::numeric, 2)                                               AS distancia_metros
FROM reportes
WHERE ST_DWithin(
    coordenadas_gps::geography,
    ST_SetSRID(ST_MakePoint(-66.8541, -29.4967), 4326)::geography,
    500
)
ORDER BY distancia_metros;

In [ ]:
%%sql
-- Cantidad de reportes por zona (mapa de calor agrupado)
SELECT
    ti.descripcion        AS tipo,
    COUNT(*)              AS total,
    estado_resolucion
FROM reportes r
JOIN tipos_infraccion ti ON ti.id_tipo = r.id_tipo
GROUP BY ti.descripcion, r.estado_resolucion
ORDER BY total DESC;

## 6. Consultas sobre evidencias

In [ ]:
%%sql
-- Evidencias con su reporte y patente asociada
SELECT
    e.id_evidencia,
    r.patente_vehiculo,
    e.url_archivo_s3,
    LEFT(e.hash_seguridad_sha256, 16) || '...' AS hash_preview
FROM evidencias e
JOIN reportes r ON r.id_reporte = e.id_reporte
ORDER BY e.id_evidencia;

## 7. INSERT / UPDATE / DELETE desde SQL puro

In [ ]:
%%sql
-- Insertar un usuario de prueba directo desde SQL
INSERT INTO usuarios (nombre_completo, dni, email, reputacion_score)
VALUES ('Carlos Test', '99999999', 'carlos@test.com', 100)
ON CONFLICT (dni) DO NOTHING
RETURNING id_usuario, nombre_completo;

In [ ]:
%%sql
-- Cambiar estado de un reporte (reemplazá el id según tu dato)
UPDATE reportes
SET estado_resolucion = 'Aprobado'
WHERE id_reporte = 1
RETURNING id_reporte, estado_resolucion;

In [ ]:
%%sql
-- Eliminar el usuario de prueba
DELETE FROM usuarios
WHERE dni = '99999999'
RETURNING id_usuario, nombre_completo;

## 8. Resultado como DataFrame de pandas
Útil para analizar datos o graficar.

In [ ]:
resultado = %sql SELECT * FROM reportes
df = resultado.DataFrame()
print(df.dtypes)
df.head()